# Isotropy Cliffs: a hands-on guide

In this notebook we reproduce the core analysis of the paper on a small scale. The recipe:

1. **Data and prompt**: load OpenBookQA and turn each question into a multiple-choice prompt.
2. **Forward pass**: run **Qwen3-8B** and capture the hidden state of the *last prompt token* at **every layer**. This is the point where the model has to decide on the answer letter. The result is an array of shape `(n_samples, n_layers, hidden_dim)`.
3. **Metrics per layer**: **IsoScore** (how uniformly variance is spread), **k-NN purity** (do points of the same answer cluster?) and **CKA** between neighbouring layers (how much does the representation change?).
4. **Find the cliff**: the layer where IsoScore drops sharply, k-NN purity jumps and CKA dips, all at the same place. Then look at it with t-SNE.
5. **Extend**: other models, datasets, prompt formats and shot counts.

Everything you may want to change lives in **one config cell** (Section 1). The metric code is in `geometric_metrics.py`, which must sit next to this notebook.

**Requirements**: a GPU with about 10 GB for Qwen3-8B in 8-bit (the default) or about 16 GB in float16. For a quick smoke test on a small GPU or CPU, set `MODEL_NAME = "Qwen/Qwen2.5-3B"` and `SAMPLE_SIZE = 2100`.

## 0. Setup

Install the dependencies once (skip this if your environment already has them) and import everything we need.

In [ ]:
# Run once. `bitsandbytes` is only needed for 8-bit loading (the default below).
%pip install -q torch transformers accelerate datasets scikit-learn scikit-dimension matplotlib seaborn tqdm scipy pandas bitsandbytes

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import torch
from datasets import concatenate_datasets, load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import geometric_metrics as gm  # IsoScore, k-NN purity, CKA, plots (the file next to this notebook)

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 1. Configuration

This is the only cell you normally need to edit.

* **`SAMPLE_SIZE`**: how many questions to embed. Memory for the embeddings is `SAMPLE_SIZE × n_layers × hidden_dim × 4 bytes` (about 2.9 GB for 5,000 samples with Qwen3-8B).
* **`PROMPT_FORMAT`**: one of the four formats from Appendix A of the paper (0 = standard block, 1 = inline prose, 2 = delimited, 3 = compact block).
* **`NUM_SHOTS`**: number of in-context examples placed before each question (0 = zero-shot).

> **A note on `SAMPLE_SIZE` and IsoScore.** IsoScore is computed exactly as in the paper's scripts: each layer is passed as `(hidden_dim, n_samples)` and the PCA uses `min(n_samples, hidden_dim)` components. With `SAMPLE_SIZE` ≥ the hidden size (4096 for Qwen3-8B, 2048 for Qwen2.5-3B; the paper uses about 5,850 samples) this is the standard setting. With fewer samples the PCA has fewer components than dimensions, a warning is printed, and the values **change substantially** and depend on the sample size, so they are **not comparable** to the paper. For a cheap test, use a smaller model (Qwen3-0.6B has hidden size 1024) with `SAMPLE_SIZE = 1500`.

In [ ]:
# ---- Model ----------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen3-8B"      # e.g. "Qwen/Qwen3-0.6B" for a quick test, "meta-llama/Llama-3.1-8B" (needs HF login)
LOAD_IN_8BIT = True               # as in the paper's scripts. False loads float16 instead (needs about 16 GB for an 8B model)
BATCH_SIZE = 16                   # lower this if you run out of GPU memory

# ---- Data and prompt ---------------------------------------------------------------------
DATASET = "openbookqa"            # "openbookqa" or "arc_easy" (add more in DATASETS below)
SAMPLE_SIZE = 3000                # number of evaluated questions; use >= hidden size for paper-comparable IsoScore (4096 for Qwen3-8B)
PROMPT_FORMAT = 0                 # 0, 1, 2 or 3
NUM_SHOTS = 0                     # 0 = zero-shot, 4 = 4-shot ICL, ...
SEED = 22

# ---- Analysis ------------------------------------------------------------------------------
KNN_K = 5                         # neighbours for k-NN purity

OUT_DIR = Path("results") / MODEL_NAME.replace("/", "_") / DATASET / f"format{PROMPT_FORMAT}_{NUM_SHOTS}shot"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs go to:", OUT_DIR)

## 2. Load the dataset

Every dataset is converted to the same small structure: a question, the answer **labels** (`A`, `B`, ...), the answer **choices** (texts), and the correct **answer** label. To use another multiple-choice dataset, add an entry to `DATASETS` with the right column names. Datasets whose columns look like `choices = {"label": [...], "text": [...]}` and `answerKey` work as they are.

We pool all splits and shuffle with a fixed seed, so the sample is reproducible.

In [ ]:
DATASETS = {
    "openbookqa": dict(path="allenai/openbookqa", name="main",     question_key="question_stem"),
    "arc_easy":   dict(path="allenai/ai2_arc",    name="ARC-Easy", question_key="question"),
}


def load_examples(dataset_name, n, seed):
    cfg = DATASETS[dataset_name]
    parts = [load_dataset(cfg["path"], cfg["name"], split=s) for s in ("train", "validation", "test")]
    ds = concatenate_datasets(parts).shuffle(seed=seed)
    ds = ds.select(range(min(n, len(ds))))
    return [
        dict(question=r[cfg["question_key"]], labels=r["choices"]["label"], choices=r["choices"]["text"], answer=r["answerKey"])
        for r in ds
    ]


examples = load_examples(DATASET, SAMPLE_SIZE + NUM_SHOTS, SEED)
shots, eval_examples = examples[:NUM_SHOTS], examples[NUM_SHOTS:]  # shots are never evaluated
print(f"{len(eval_examples)} evaluation examples, {len(shots)} in-context examples")
eval_examples[0]

## 3. Build the prompts

The four formats carry the same content with different surface syntax (Appendix A of the paper). Every prompt **ends right where the answer letter should come next**. That last token is where we will read the representation.

With `NUM_SHOTS > 0`, solved examples in the same format are placed in front of each question.

In [ ]:
def format_question(ex, fmt):
    q, labels, choices = ex["question"], ex["labels"], ex["choices"]
    pairs = list(zip(labels, choices))
    if fmt == 0:   # standard block layout
        body = "\n".join(f"{l}: {c}" for l, c in pairs)
        return f"Question: {q}\n\nChoices:\n{body}\n\nAnswer:"
    if fmt == 1:   # continuous prose / inline
        body = ", ".join(f"({l}) {c}" for l, c in pairs)
        return f"{q}\nChoose from: {body}. The answer is"
    if fmt == 2:   # highly delimited / machine-readable
        body = "; ".join(f"{l}={c}" for l, c in pairs)
        return f"QUESTION: {q} || CHOICES: {body} || ANSWER:"
    if fmt == 3:   # compact block
        body = "\n".join(f"{l}: {c}" for l, c in pairs)
        return f"Question: {q}\nChoices: {body}\nAnswer:"
    raise ValueError(f"Unknown prompt format {fmt}")


def build_prompts(eval_examples, shots, fmt):
    context = "".join(format_question(s, fmt) + f" {s['answer']}\n\n" for s in shots)
    return [context + format_question(ex, fmt) for ex in eval_examples]


prompts = build_prompts(eval_examples, shots, PROMPT_FORMAT)
print(prompts[0])

## 4. Load the model

We load Qwen3-8B in 8-bit (as in the paper's scripts) and put it on the GPU. **Precision matters for IsoScore.** Hidden states of LLMs have a few dimensions with huge magnitudes, and bfloat16 rounds such values coarsely, which adds noise that can distort variance-based metrics. In a toy test with such offsets, rounding to bf16 lowered IsoScore about 12× while float16 barely changed it. So use 8-bit (computes in float16) or float16/float32, **not bf16**. If float16 produces `nan` for some model, use float32 for small models.

**Left padding** is important: it makes the last prompt token the last position for every sample in a batch, so `hidden_state[:, -1, :]` is always the "about to answer" position.

In [ ]:
def load_model(model_name, in_8bit=True, dtype=torch.float16):
    kwargs = dict(device_map="auto")
    if in_8bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)  # computes in float16
    else:
        kwargs["torch_dtype"] = dtype  # float16 or float32; avoid bfloat16 (see the note above)
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs).eval()
    model.config.use_cache = False
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


model, tokenizer = load_model(MODEL_NAME, LOAD_IN_8BIT)
n_layers = model.config.num_hidden_layers
print(f"{MODEL_NAME}: {n_layers} layers, hidden size {model.config.hidden_size}")

## 5. Forward pass: capture embeddings and predictions

For every batch we do two things:

* **Embeddings**: `output_hidden_states=True` returns the activations after every layer. We keep only the **last token's** vector of each layer. `hidden_states[0]` is the embedding layer and is skipped, so layer `0` below is the output of the first transformer block. The final entry is after the model's last norm.
* **Prediction**: from the next-token logits we compare the probabilities of the answer letters (with and without a leading space, since tokenizers differ) and pick the most likely one. This gives us the model's accuracy.

The result is stored as `embeddings` with shape `(n_samples, n_layers, hidden_dim)`.

In [ ]:
@torch.inference_mode()
def extract(model, tokenizer, prompts, examples, batch_size):
    id_cache = {}

    def letter_ids(label):  # token ids for "A" and " A"
        if label not in id_cache:
            id_cache[label] = [tokenizer.convert_tokens_to_ids(label), tokenizer.encode(" " + label, add_special_tokens=False)[0]]
        return id_cache[label]

    embeddings, predictions = [], []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Forward pass"):
        batch = tokenizer(prompts[i : i + batch_size], return_tensors="pt", padding=True).to(model.device)
        out = model(**batch, output_hidden_states=True)

        # (batch, n_layers, hidden_dim): last-token vector at every layer
        hidden = torch.stack([h[:, -1, :] for h in out.hidden_states[1:]], dim=1)
        embeddings.append(hidden.float().cpu().numpy())

        probs = out.logits[:, -1, :].float().softmax(-1).cpu().numpy()
        for p, ex in zip(probs, examples[i : i + batch_size]):
            letter_probs = [max(p[t] for t in letter_ids(l)) for l in ex["labels"]]
            predictions.append(ex["labels"][int(np.argmax(letter_probs))])

    return np.concatenate(embeddings), np.array(predictions)


embeddings, predictions = extract(model, tokenizer, prompts, eval_examples, BATCH_SIZE)
labels = np.array([ex["answer"] for ex in eval_examples])
accuracy = float((predictions == labels).mean())
print(f"embeddings: {embeddings.shape}  (samples, layers, hidden_dim)")
print(f"accuracy:   {accuracy:.3f}")

Save the embeddings so you never need to repeat the expensive forward pass. Then free the GPU, since the rest of the notebook only needs the saved array.

In [ ]:
np.save(OUT_DIR / "embeddings.npy", embeddings)
np.save(OUT_DIR / "labels.npy", labels)
np.save(OUT_DIR / "predictions.npy", predictions)
json.dump(dict(model=MODEL_NAME, dataset=DATASET, format=PROMPT_FORMAT, shots=NUM_SHOTS, n=len(labels), accuracy=accuracy),
          open(OUT_DIR / "config.json", "w"), indent=2)

del model
gc.collect()
torch.cuda.empty_cache()

*(Optional)* In a later session, skip Sections 4 and 5 and reload the saved arrays instead:

In [ ]:
# embeddings = np.load(OUT_DIR / "embeddings.npy"); labels = np.load(OUT_DIR / "labels.npy"); predictions = np.load(OUT_DIR / "predictions.npy")
# accuracy = float((predictions == labels).mean())

## 6. IsoScore per layer

**IsoScore** measures how uniformly the variance of a point cloud is spread over directions. After rotating the data into its PCA basis, it compares the variance vector with the perfectly uniform one. `1` means a round, uniform cloud. A value near `0` means the variance is concentrated in a few directions.

We compute it on all samples of one layer at a time (the cloud is `n_samples` points in `hidden_dim` dimensions), using the reference implementation from the paper. `gm.isoscore` passes each layer as `(hidden_dim, n_samples)`, i.e. the same as `IsoScore(layer.T)` in the original scripts, and uses `min(n_samples, hidden_dim)` PCA components, exactly as in the original scripts (see the note in Section 1). It can take a few minutes for 5,000+ samples, because every layer needs a full PCA. What to expect: IsoScore rises through the early layers, peaks, and then **drops sharply** at the cliff.

In [ ]:
n_samples, n_layers, hidden_dim = embeddings.shape
print(f"{n_samples} samples, hidden size {hidden_dim}")  # for paper-comparable IsoScores use n_samples >= hidden_dim (see Section 1)

iso = gm.isoscore_per_layer(embeddings)
gm.plot_layerwise(iso, "IsoScore", f"IsoScore per layer: {MODEL_NAME}", color=gm.C_ISO, save_path=OUT_DIR / "isoscore.png")

## 7. k-NN purity per layer

**k-NN purity** asks whether neighbouring points share the same *ground-truth answer letter*: for each sample we look at its `k` nearest neighbours (Euclidean) and take the fraction with the same label, then average over samples.

With 4 balanced answer options, a representation without answer information scores about **0.25**. A score near 1 means the answers form clean clusters. What to expect: close to chance in early layers, then a **sudden jump** at the cliff.

In [ ]:
knn = gm.knn_purity_per_layer(embeddings, labels, k=KNN_K)
print("chance level (largest class share):", round(float(np.unique(labels, return_counts=True)[1].max() / len(labels)), 3))
gm.plot_layerwise(knn, "k-NN purity", f"k-NN purity per layer: {MODEL_NAME}", color=gm.C_KNN, save_path=OUT_DIR / "knn_purity.png", ylim=(0, 1.05))

## 8. CKA between neighbouring layers

**Linear CKA** compares two representations of the same samples and is invariant to rotation and scaling. `1` means "the geometry is unchanged", and low values mean "the representation was substantially reorganised". We compute it between each layer `l` and the next, `l+1`. Entry `i` of the result is `CKA(layer i, layer i+1)`.

What to expect: values near 1 almost everywhere, with a **dip** at the cliff. The last layers can also change a lot, so a dip at the very end is usually unrelated.

In [ ]:
cka = gm.cka_neighboring_layers(embeddings)  # shape (n_layers - 1,); uses the GPU if available
gm.plot_layerwise(cka, "CKA(l, l+1)", f"CKA between neighbouring layers: {MODEL_NAME}", color=gm.C_CKA, save_path=OUT_DIR / "cka.png", ylim=(0, 1.05))

## 9. Putting it together: where is the cliff?

`find_cliff` returns, for each signal, the transition `L → L+1` where it changes the most:

* `iso_drop`: the largest drop in IsoScore
* `knn_jump`: the largest increase in k-NN purity
* `cka_dip`: the lowest neighbouring-layer CKA

If the three agree, you have found the isotropy cliff. For Qwen3-8B on OpenBookQA the paper reports the transition **23 → 24**. Your numbers can shift by a layer with a different sample or prompt. This is a simple heuristic that ignores the last two transitions, so **always check the plot**; you can override `L_CLIFF` by hand.

We also compute the cliff's steepness, **ΔLog IsoScore** `= log IsoScore(L+1) − log IsoScore(L)` (more negative = steeper). Across many configurations (Section 12) this value is what correlates with accuracy.

In [ ]:
cliff = gm.find_cliff(iso, knn, cka)
print(cliff)

L_CLIFF = cliff["iso_drop"]          # override by hand if the plot tells you otherwise, e.g. L_CLIFF = 23
d_li = gm.delta_log_iso(iso, L_CLIFF)
print(f"Cliff at layers {L_CLIFF} -> {L_CLIFF + 1}")
print(f"  IsoScore  {iso[L_CLIFF]:.4f} -> {iso[L_CLIFF + 1]:.4f}   (ΔLog IsoScore = {d_li:+.3f})")
print(f"  k-NN      {knn[L_CLIFF]:.3f} -> {knn[L_CLIFF + 1]:.3f}")
print(f"  CKA       {cka[L_CLIFF]:.3f}")

gm.plot_cliff_overview(iso, knn, cka, cliff_layer=L_CLIFF, title=f"{MODEL_NAME} on {DATASET} (accuracy {accuracy:.2f})",
                       save_path=OUT_DIR / "cliff_overview.png")

## 10. Seeing the clusters with t-SNE

Finally, project the layers just before and after the cliff to 2D and colour by the correct answer. Before the cliff the colours should be mixed. Right after it they separate into clusters.

Each t-SNE run is independent, so positions are **not comparable between the two panels**; only the cluster structure within each panel matters. This cell takes a minute or so for a few thousand samples.

In [ ]:
gm.plot_tsne_pair(embeddings, labels, layers=[L_CLIFF, L_CLIFF + 1], save_path=OUT_DIR / "tsne_cliff.png")

## 11. Optional: intrinsic dimension

The paper contrasts the isotropy cliff with **intrinsic dimension (ID)**, which rises and then compresses *gradually* instead of dropping abruptly. This uses the TwoNN estimator and can be slow for large samples.

In [ ]:
# ids = gm.intrinsic_dim_per_layer(embeddings)
# gm.plot_layerwise(ids, "TwoNN intrinsic dimension", f"Intrinsic dimension: {MODEL_NAME}", color="#16a34a", save_path=OUT_DIR / "twonn.png")

## 12. Extending the experiment

Everything above is one **(model, dataset, prompt format, shots)** configuration. The paper's main result comes from many of them: the steeper the cliff (more negative ΔLog IsoScore), the higher the accuracy.

The cell below runs the whole pipeline in a loop and collects one row per configuration. It is switched **off** by default, because every extra configuration costs a forward pass. To use it:

* **More models**: add names to `MODELS` (gated ones like Llama need `huggingface_hub.login()` first).
* **More prompt formats / shots**: change `FORMATS` and `SHOTS`.
* **More datasets**: add an entry to `DATASETS` in Section 2 and list it in `DATASET_NAMES`.

Afterwards, the last cell computes the Spearman correlation between ΔLog IsoScore and accuracy, as in Figure 2 of the paper. It needs at least about 10 configurations to be meaningful.

In [ ]:
RUN_SWEEP = False   # set to True to run

MODELS = ["Qwen/Qwen3-8B"]          # e.g. add "Qwen/Qwen2.5-3B", "mistralai/Mistral-7B-v0.3"
DATASET_NAMES = ["openbookqa"]
FORMATS = [0, 1, 2, 3]
SHOTS = [0, 2, 4]

results = []


def analyse(embeddings, labels, k=KNN_K):
    iso = gm.isoscore_per_layer(embeddings)
    knn = gm.knn_purity_per_layer(embeddings, labels, k)
    cka = gm.cka_neighboring_layers(embeddings)
    cliff = gm.find_cliff(iso, knn, cka)
    return dict(iso=iso, knn=knn, cka=cka, cliff=cliff["iso_drop"], d_li=gm.delta_log_iso(iso, cliff["iso_drop"]))


if RUN_SWEEP:
    for model_name in MODELS:
        model, tokenizer = load_model(model_name, LOAD_IN_8BIT)
        for ds_name in DATASET_NAMES:
            pool = load_examples(ds_name, SAMPLE_SIZE + max(SHOTS), SEED)
            for n_shots in SHOTS:
                shots_, evals_ = pool[:n_shots], pool[max(SHOTS):]       # same evaluation set for every shot count
                for fmt in FORMATS:
                    emb, preds = extract(model, tokenizer, build_prompts(evals_, shots_, fmt), evals_, BATCH_SIZE)
                    lab = np.array([ex["answer"] for ex in evals_])
                    res = analyse(emb, lab)
                    res.update(model=model_name, dataset=ds_name, fmt=fmt, shots=n_shots, accuracy=float((preds == lab).mean()))
                    results.append({k: v for k, v in res.items() if k not in ("iso", "knn", "cka")})
                    print(results[-1])
        del model
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

if results:
    df = pd.DataFrame(results)
    df.to_csv("sweep_results.csv", index=False)
    rho, p = spearmanr(df["d_li"], df["accuracy"])
    print(f"Spearman rho(ΔLog IsoScore, accuracy) = {rho:.2f}  (p = {p:.3g}, n = {len(df)})")
    df.sort_values("accuracy", ascending=False).head(10)